# Likelihood and MAP estimate for a single parameter

Illustration of the sequential Bayesian update for one parameter. The Clear Creek stream network and radar cells are used with synthetic storms generated from known parameters, so the posterior can be compared with the true value.

Only the channel celerity $c$ is estimated; all other parameters are fixed at their true values. The calibration in `clear_creek_bayes.ipynb` applies the same procedure to all nine parameters and each storm's $I_a$ jointly.

Panels: (a) storm 1 simulated with three values of $c$; (b) residuals $r_j = (\mathrm{sim}_j - \mathrm{obs}_j)/\mathrm{peak}$ and their sum of squares; (c) $\sum r_j^2$ as a function of $c$ and the likelihood $\propto \exp(-\sum r_j^2 / 2s^2)$; (d) prior, likelihood and posterior for storm 1, with the MAP estimate; (e) the same for storm 2, with the storm 1 posterior as prior.

Required: `distributed_iuh.py`, `sequential_bayes.py` and `nash_iuh.py` in this folder, and the basin files written by sections 1–3 of `clear_creek_bayes.ipynb`.

## 1. Settings

In [ ]:
import dataclasses
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import distributed_iuh as d
import sequential_bayes as sb

mpl.rcParams["pdf.fonttype"] = 42          # TrueType fonts in PDF output

DATA = "data/USGS-05454300"                 # data folder of clear_creek_bayes.ipynb
DT = 0.25                                   # time step (h)
KB_DAYS = 20.0                              # baseflow recession constant (days)

# true parameters of the synthetic storms
TRUE = d.DistParams(0.9, 600, 3.0, 0.12, 0.03, beta=0.15, k_gw_h=60.0)   # c, D, k_hill, fc, a

PRIOR_MEAN, PRIOR_SD = 1.0, 0.6             # prior on c (m/s), as in the calibration notebooks
C_GRID = np.linspace(0.25, 2.2, 400)        # evaluation grid for c (m/s)
SHOW_C = [0.45, None, 1.80]                 # values shown in (a)-(c); None = storm 1 MAP

NOISE_AR, NOISE_SD = 0.97, 0.035            # AR(1) coefficient and sd of the multiplicative flow error
SEED = 11

SAVE_PNG = f"{DATA}/likelihood_MAP_explained.png"
SAVE_PDF = f"{DATA}/likelihood_MAP_explained.pdf"   # vector output

## 2. Basin and router

In [ ]:
lines = d.load_lines(f"{DATA}/nldi_flowlines.geojson")
ring = d.load_ring(f"{DATA}/nldi_basin.geojson")
outlet = d.load_point(f"{DATA}/nldi_site.geojson")
net = d.build_channel_network(lines, outlet)
cube = d.RadarCube.load(f"{DATA}/mrms_basin.npz")          # radar grid only
cells = d.basin_cells(cube.lon1d, cube.lat1d, ring, net)
router = d.Router(cells, DT)
print(f"{len(cells)} radar cells, {cells.area_mi2.sum():.1f} mi²")

## 3. Synthetic storms

Each storm has constant rainfall intensity and a Gaussian spatial distribution (standard deviation 7 km), and is routed with the true parameters. The synthetic observed flow is the simulated hydrograph multiplied by $(1 + e_t)$, where $e_t$ is an AR(1) process. The resulting residuals are autocorrelated, as in observed data.

In [ ]:
rng = np.random.default_rng(SEED)

def make_storm(day, centre, ia, depth_in_per_step, duration_h):
    n = int(72 / DT)
    hy = np.zeros(n)
    s0 = int(4 / DT)
    hy[s0:s0 + int(duration_h / DT)] = depth_in_per_step
    P = d.storm_at(cells, centre, 7, hy)                       # rain on every cell
    pe = d.cell_excess(P, TRUE, ia, DT)
    q = (router.route(pe, TRUE, length=n)
         + d.gw_return_flow(d.infiltration(P, pe, cells.area_mi2.values), TRUE, DT, n)
         + d.recession(30, n, DT, KB_DAYS))
    e = np.zeros(n)
    for j in range(1, n):
        e[j] = NOISE_AR * e[j - 1] + rng.standard_normal() * NOISE_SD
    q = q * (1 + e)
    t = pd.date_range("2020-06-01", periods=n, freq="15min") + pd.Timedelta(days=day)
    return d.DistEvent(t, P, q, np.zeros(n), q, 0.0), ia

lon0, lon1 = cells.lon.min(), cells.lon.max()
lat0, lat1 = cells.lat.min(), cells.lat.max()
ev1, ia1 = make_storm(0,  (lon0 + 0.7 * (lon1 - lon0), lat0 + 0.5 * (lat1 - lat0)), 0.35, 0.15, 4)
ev2, ia2 = make_storm(20, (lon0 + 0.2 * (lon1 - lon0), lat0 + 0.6 * (lat1 - lat0)), 0.45, 0.12, 5)

## 4. Residuals, likelihood and posterior on a grid of $c$

For each value of $c$ the model is run and the sum of squared residuals is computed. The noise variance $s^2 = \sigma^2(1+\rho)/(1-\rho)$ is estimated from the residuals at the best $c$ with `sb._noise`, as in the calibration. The posterior is normalised on the grid. Its maximum is the MAP estimate, and the curvature of the log posterior at the maximum gives the Laplace (normal) standard deviation, which defines the prior for the next storm.

In [ ]:
def residuals(ev, ia, c):
    p = dataclasses.replace(TRUE, c_ms=c)
    sim, target, rec = sb.simulate(ev, p, ia, router, DT, KB_DAYS)
    return (sim - target) / target.max(), sim, target, rec

def log_likelihood(ev, ia):
    R = [residuals(ev, ia, c)[0] for c in C_GRID]
    ss = np.array([np.sum(r ** 2) for r in R])
    s2, sig2, rho = sb._noise(R[int(np.argmin(ss))])
    return -0.5 * ss / s2, ss, s2, sig2, rho

def normalise(logp):
    p = np.exp(logp - logp.max())
    return p / np.trapezoid(p, C_GRID)

def laplace(logpost):
    i = int(np.argmax(logpost))
    h = C_GRID[1] - C_GRID[0]
    curv = -(logpost[i + 1] - 2 * logpost[i] + logpost[i - 1]) / h ** 2
    return C_GRID[i], 1 / np.sqrt(curv)

# storm 1: prior from the settings
logprior1 = -0.5 * ((C_GRID - PRIOR_MEAN) / PRIOR_SD) ** 2
ll1, ss1, s2_1, sig2_1, rho1 = log_likelihood(ev1, ia1)
m1, sd1 = laplace(ll1 + logprior1)

# storm 2: storm 1's posterior (as a normal) is the prior
logprior2 = -0.5 * ((C_GRID - m1) / sd1) ** 2
ll2, ss2, s2_2, sig2_2, rho2 = log_likelihood(ev2, ia2)
m2, sd2 = laplace(ll2 + logprior2)

print(f"true c = {TRUE.c_ms}")
print(f"after storm 1: MAP {m1:.3f} m/s, sd {sd1:.3f}  (rho = {rho1:.2f})")
print(f"after storm 2: MAP {m2:.3f} m/s, sd {sd2:.3f}  (rho = {rho2:.2f})")

## 5. Figure

In [ ]:
C_PRI, C_LIK, C_POS = "#7570b3", "#d95f02", "#c51b7d"
show = list(zip([m1 if c is None else c for c in SHOW_C], ["#4575b4", "#1a9850", "#d73027"]))
c_bad = show[-1][0]

fig = plt.figure(figsize=(16, 12.5))
gs = fig.add_gridspec(3, 2, height_ratios=[1, 0.8, 1.05], hspace=0.5, wspace=0.18)
hrs = (ev1.time - ev1.time[0]) / pd.Timedelta(hours=1)

# (a) hydrographs
ax = fig.add_subplot(gs[0, 0])
_, _, target, _ = residuals(ev1, ia1, 1.0)
ax.plot(hrs, target, "k", lw=2.2, label="observed event flow")
for c, col in show:
    ax.plot(hrs, residuals(ev1, ia1, c)[1], color=col, lw=1.6, label=f"simulated, c = {c:.2f} m/s")
ax.fill_between(hrs, residuals(ev1, ia1, c_bad)[1], target, color=show[-1][1], alpha=0.15,
                label=f"residuals for c = {c_bad:.2f}")
ax.set(xlim=(0, 48), xlabel="Time (h)", ylabel="Event flow (cfs)")
ax.set_title("(a) Storm 1: simulations for three values of c", loc="left", fontsize=12)
ax.legend(fontsize=8.5, frameon=False)

# (b) residuals
ax = fig.add_subplot(gs[0, 1])
for c, col in show:
    r = residuals(ev1, ia1, c)[0]
    ax.plot(hrs, r, color=col, lw=1.5, label=f"c = {c:.2f}:  Σ r² = {np.sum(r ** 2):.2f}")
ax.axhline(0, color="k", lw=0.8)
ax.set(xlim=(0, 48), xlabel="Time (h)", ylabel="Residual $r_j$ (fraction of observed peak)")
ax.set_title("(b) Residuals and sum of squares", loc="left", fontsize=12)
ax.legend(fontsize=9, frameon=False)

# (c) sum of squares over c
ax = fig.add_subplot(gs[1, :])
ax.plot(C_GRID, ss1, color=C_LIK, lw=2.2)
for c, col in show:
    j = int(np.argmin(abs(C_GRID - c)))
    ax.plot(C_GRID[j], ss1[j], "o", color=col, ms=9, mec="k", zorder=5)
ax.set(xlabel="c (m/s)", ylabel="Σ r² (storm 1)", xlim=(C_GRID[0], C_GRID[-1]))
ax.set_title("(c) Sum of squared residuals as a function of c",
             loc="left", fontsize=12)
ax.text(0.5, 0.93, f"likelihood  ∝  exp( −Σ r² / 2s² )\n"
        f"s² = σ² (1+ρ)/(1−ρ):  σ² = {sig2_1:.4f},  ρ = {rho1:.2f} (lag-1 autocorrelation),  "
        f"(1+ρ)/(1−ρ) = {(1 + rho1) / (1 - rho1):.0f}",
        transform=ax.transAxes, ha="center", va="top", fontsize=10.5,
        bbox=dict(boxstyle="round,pad=0.4", fc="w", ec=C_LIK))
ax.grid(alpha=0.3)

# (d), (e) prior x likelihood = posterior
for pos_, lp_prior, ll, mm, sdd, title in [
        (gs[2, 0], logprior1, ll1, m1, sd1, "(d) Storm 1: prior × likelihood = posterior"),
        (gs[2, 1], logprior2, ll2, m2, sd2, "(e) Storm 2: prior = storm 1 posterior")]:
    ax = fig.add_subplot(pos_)
    pri, lik, post = normalise(lp_prior), normalise(ll), normalise(lp_prior + ll)
    ax.fill_between(C_GRID, post, color=C_POS, alpha=0.18)
    ax.plot(C_GRID, pri, color=C_PRI, lw=2, ls="--", label="prior")
    ax.plot(C_GRID, lik, color=C_LIK, lw=2, label="likelihood")
    ax.plot(C_GRID, post, color=C_POS, lw=2.6, label="posterior ∝ prior × likelihood")
    ax.axvline(mm, color=C_POS, lw=1, ls=":")
    lo, hi = mm - 1.645 * sdd, mm + 1.645 * sdd
    ax.axvspan(lo, hi, color=C_POS, alpha=0.07)
    top = max(post.max(), lik.max())
    ax.annotate(f"MAP: c = {mm:.2f} m/s\n90 % interval: {lo:.2f}–{hi:.2f}",
                xy=(mm, post.max()), xytext=(mm + 0.18, top * 0.45), fontsize=9.5,
                arrowprops=dict(arrowstyle="->", color="0.3"),
                bbox=dict(boxstyle="round,pad=0.3", fc="w", ec="0.7"))
    ax.set(xlabel="c (m/s)", ylabel="Probability density", xlim=(0.4, 1.6), ylim=(0, top * 1.12))
    ax.set_title(title, loc="left", fontsize=12)
    ax.legend(fontsize=8.5, frameon=False, loc="upper left")

fig.savefig(SAVE_PNG, dpi=200, bbox_inches="tight")
fig.savefig(SAVE_PDF, bbox_inches="tight")
plt.show()